# 時系列データの処理
時間の順に並んだデータを、成長率・幾何平均・移動平均・自己相関(コレログラム)で分析します。トレンドと季節変動を見分け、平均成長率や移動平均を手計算で求められるようにします。

## 学習目標
- 時系列データをトレンド・季節変動・不規則変動に分けて捉えられる
- 前期比・前年同期比などの成長率を計算し、使い分けられる
- 平均成長率を幾何平均で求め、算術平均との違いを説明できる
- 奇数項・偶数項(中心化)の移動平均を手計算できる
- 自己相関係数の定義を理解し、コレログラムからトレンドや周期を読み取れる

## 時系列データの構成要素

時系列データは、一般に次の要素が重なったものと考えます。

| 要素 | 内容 | 例 |
|---|---|---|
| **トレンド**(傾向変動) | 長期的な増加・減少の傾向 | 人口の減少、売上の成長 |
| **季節変動** | 1年などの周期で繰り返す変動 | 夏と年末に売上が増える |
| 循環変動 | 数年単位の景気の波 | |
| 不規則変動 | 上記で説明できない偶然の変動 | |

次の図は、ある会社の月次売上(2018〜2025 年)です。細い線が実際の売上、太い線がトレンド(後で説明する 12 か月の中心化移動平均)です。

![月次売上とトレンド](images/u09_series.png)

右上がりのトレンドに、毎年同じ時期(夏と 12 月)に山ができる季節変動が重なっていることがわかります。

## 成長率(変化率)

$$\text{成長率}_t = \frac{x_t - x_{t-1}}{x_{t-1}} = \frac{x_t}{x_{t-1}} - 1$$

- **前期比**(前月比):直前の時点と比べる。季節変動の影響を強く受ける
- **前年同期比**(前年同月比):1年前の同じ月と比べる。**季節変動の影響を除いて** 傾向を見られる

**例**:上の月次売上の一部

| 月 | 売上(百万円) | 前月比 | 前年同月比 |
|---|---|---|---|
| 2024 年 11 月 | 201.7 | +8.1% | +11.4% |
| 2024 年 12 月 | 217.8 | +8.0% | +7.2% |
| 2025 年 1 月 | 190.7 | −12.4% | +3.6% |
| 2025 年 2 月 | 190.3 | −0.2% | +13.3% |

2025 年 1 月の前月比は −12.4% と大きく下がっていますが、これは 12 月の売上が季節的に多いためです。前年同月比で見ると +3.6% で、売上は前年より増えています。

## 平均成長率と幾何平均

成長率は **掛け算で積み重なる** ため、平均的な成長率は算術平均ではなく **幾何平均** で求めます。

$$1 + \bar{g} = \sqrt[n]{(1 + g_1)(1 + g_2)\cdots(1 + g_n)} = \left(\frac{x_n}{x_0}\right)^{1/n}$$

**例**:1年目 +50%、2年目 −50% のとき

- 算術平均は (50 − 50) ÷ 2 = 0% だが、実際は $1.5 \times 0.5 = 0.75$ 倍(25% 減)
- 幾何平均では $\sqrt{0.75} - 1 \approx -0.134$、つまり **年平均 −13.4%** が正しい平均成長率

一般に、幾何平均 ≦ 算術平均 が成り立ちます(すべて等しいときだけ一致)。
上の会社の年間売上(2018 年 1506.9 → 2025 年 2515.5)の 7 年間の平均成長率は、$(2515.5 \div 1506.9)^{1/7} - 1 \approx 7.6\%$ です。

## 移動平均による平滑化

前後の値の平均をとって、不規則変動や季節変動をならす方法を **移動平均** といいます。

**奇数項**($2k+1$ 項)の移動平均は、時点 $t$ を中心に前後 $k$ 個ずつの平均です。

**例**:7 個のデータ `12, 15, 18, 14, 16, 20, 17` の 3 項移動平均

| 時点 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|
| 値 | 12 | 15 | 18 | 14 | 16 | 20 | 17 |
| 3 項移動平均 | − | 15.0 | 15.7 | 16.0 | 16.7 | 17.7 | − |

時点 2 の値は (12 + 15 + 18) ÷ 3 = 15.0 です。両端の時点は前後の値がそろわないので求まりません。

**偶数項** の移動平均(例:4 項、12 項)は、平均の中心が時点と時点の間にずれるため、さらに隣どうしを平均して中心をそろえます(**中心化移動平均**)。結果として、両端に 1/2 の重みをつけた平均になります。

$$\text{4 項の中心化移動平均:}\quad \frac{\tfrac{1}{2}x_{t-2} + x_{t-1} + x_t + x_{t+1} + \tfrac{1}{2}x_{t+2}}{4}$$

$$\text{12 項の中心化移動平均:}\quad \frac{\tfrac{1}{2}x_{t-6} + x_{t-5} + \cdots + x_{t+5} + \tfrac{1}{2}x_{t+6}}{12}$$

周期(月次データなら 12、四半期データなら 4)と同じ長さの移動平均をとると、**季節変動が消えて** トレンドが見えるようになります(上の図の太い線)。

## 自己相関とコレログラム

時系列で、時点 $t$ の値と $k$ 時点前の値の相関を **自己相関**(系列相関)といい、$k$ を **ラグ** といいます。統計検定などで使われる自己相関係数の定義は次のとおりです。

$$r_k = \frac{\sum_{t=1}^{n-k}(x_t - \bar{x})(x_{t+k} - \bar{x})}{\sum_{t=1}^{n}(x_t - \bar{x})^2}$$

- 分母は **系列全体** の偏差平方和、平均 $\bar{x}$ も系列全体の平均を使う
- 分子は「$k$ だけずらした偏差どうしの積」を、ずらして重なる部分だけ合計する

**例**:5 個のデータ `4, 8, 6, 10, 12`(平均 8)のラグ 1 の自己相関係数

| $t$ | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| 偏差 $x_t - \bar{x}$ | −4 | 0 | −2 | 2 | 4 |

- 分母:$16 + 0 + 4 + 4 + 16 = 40$
- 分子(隣どうしの積):$(-4)(0) + (0)(-2) + (-2)(2) + (2)(4) = 0 + 0 - 4 + 8 = 4$
- $r_1 = 4 \div 40 = 0.1$

ラグ $k = 1, 2, \dots$ に対して $r_k$ を棒グラフにしたものが **コレログラム** です。

![月次売上のコレログラム(元の系列と前月差)](images/u09_correlogram.png)

| コレログラムの形 | 読み取れること |
|---|---|
| ゆっくり減衰する正の値 | トレンドがある(左の図) |
| 周期ごとに山がある | その周期の季節変動がある(右の図では 12・24・36 か月ごと) |
| ラグ 1 以降ほぼ 0 | ランダムな系列(前の値と無関係) |

トレンドが強い系列では、どのラグでも大きな正の自己相関が出て季節変動が見えにくくなります。その場合は、前期との差(階差)をとってトレンドを除いてから見ます(右の図)。

## 演習

<!-- @question
type: numeric
answer: 10
tolerance: 0.01
title: 平均成長率を幾何平均で求める
hint: 3 年間で 133.1 ÷ 100 = 1.331 倍になっています。1 年あたり何倍になれば、3 回掛けて 1.331 になるかを考えます。
explanation: 1.331 = 1.1 × 1.1 × 1.1 なので、1 年あたり 1.1 倍、平均成長率は 10% です。
-->
ある企業の売上高は、ある年に 100 億円、その 3 年後に 133.1 億円であった。この 3 年間の年平均成長率(%)を、幾何平均を用いて求めよ。

<!-- @question
type: numeric
answer: 24
tolerance: 0.001
title: 3 項移動平均を求める
hint: 4 番目の時点の 3 項移動平均は、3・4・5 番目の値の平均です。
explanation: 4 番目の時点を中心とする 3 つの値は 20, 24, 28 なので、(20 + 24 + 28) ÷ 3 = 24 です。
-->
6 つの時点の値が `21, 25, 20, 24, 28, 30` であった。4 番目の時点(値 24)における 3 項移動平均を求めよ。

<!-- @question
type: numeric
answer: 14.25
tolerance: 0.001
title: 中心化移動平均を求める
hint: 4 項の中心化移動平均は、中心の時点の前後 2 つずつ(計 5 つ)の値を使い、両端の値だけ 1/2 の重みで足して 4 で割ります。
explanation: (0.5 × 10 + 14 + 12 + 18 + 0.5 × 16) ÷ 4 = (5 + 14 + 12 + 18 + 8) ÷ 4 = 57 ÷ 4 = 14.25 です。
-->
四半期ごとの 5 つの時点の値が `10, 14, 12, 18, 16` であった。3 番目の時点(値 12)における 4 項の中心化移動平均を求めよ。

<!-- @question
type: matrix
answer: 季節変動がある, トレンドがある, ランダム
title: コレログラムの読み取り
hint: 周期的に山と谷が繰り返すか、ゆっくり減っていくか、ラグ 1 からほぼ 0 かを見ます。
explanation: A は 12 ごとに山がくり返すので、周期 12 の季節変動がある系列です。B は大きな正の値がゆっくり減衰しているので、トレンドがある系列です。C はラグ 1 以降ほぼ 0 なので、ランダムな系列です。
-->
次の A〜C は、3 つの月次データのコレログラム(横軸:ラグ、縦軸:自己相関係数)である。

![コレログラム A〜C](images/u09_ex_correlograms.png)

それぞれの系列の特徴として最も適切なものを、行ごとに1つ選べ。

| コレログラム | トレンドがある | 季節変動がある | ランダム |
|---|---|---|---|
| A | | | |
| B | | | |
| C | | | |

<!-- @question
type: choice
answer: 平均成長率は、各期の成長率の算術平均で求めるのが正しい。
title: 時系列の指標の使い分け
hint: 本文の「平均成長率と幾何平均」の例(+50% と −50%)を思い出しましょう。
explanation: 成長率は掛け算で積み重なるので、平均成長率は幾何平均で求めます。算術平均では過大になります(+50% と −50% の例では、算術平均 0% に対して実際は 25% 減)。そのほかの記述は正しい内容です。
-->
時系列データの分析について述べた次の記述のうち、**誤っているもの** を1つ選べ。

- 前年同月比は、季節変動の影響を除いて変化を見るのに適している。
- 平均成長率は、各期の成長率の算術平均で求めるのが正しい。
- 月次データに 12 項の中心化移動平均をとると、12 か月周期の季節変動を取り除ける。
- 前月比は、季節変動の影響を強く受ける。
- 1 年目に 50% 増加し、2 年目に 50% 減少すると、最初の値より小さくなる。

## まとめ

- 時系列はトレンド・季節変動・不規則変動などの重なりとして捉える。前年同期比は季節変動の影響を除いて比べられる。
- 平均成長率は幾何平均 $\left(x_n/x_0\right)^{1/n} - 1$ で求める。算術平均は過大になる。
- 奇数項の移動平均は前後の平均。偶数項は両端に 1/2 の重みをつけた中心化移動平均にする。周期と同じ長さで季節変動を除去できる。
- 自己相関係数 $r_k$ の分母は系列全体の偏差平方和。コレログラムから、トレンド(ゆっくり減衰)や周期(周期ごとの山)を読み取る。